In [1]:
import json
import mlflow
import joblib
import onnxmltools
import numpy as np
import pandas as pd
import onnxruntime as ort

from lightgbm import LGBMClassifier
from lightgbm import early_stopping
from onnxruntime import InferenceSession
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from onnxmltools.convert.common.data_types import FloatTensorType

import sys
sys.path.append("..")
from utils import get_table, set_mlflow_experiment
from feature_engineering import time_based_split, infer_column_types, bool_to_int, datetime_to_int64, get_feature_names_from_preprocessor, get_feature_names_from_preprocessor

In [126]:
# Naming
experiment_name: str = "lightgbm_pipeline"
run_name: str = "test_1"

# Model training
train_frac: float = 0.8
val_frac: float = 0.1
drop_importance_below: float = 0.0  # es. 0.0 = niente drop, oppure 1e-6 / 0.0001
onnx_export_path: str = "artifacts/model.onnx"
preprocessor_export_path: str = "artifacts/preprocessor.joblib"

features_cols = ["chance1x2_quote_diffRealCurr1", "chance1x2_quote_diffRealCurr2", "evaluation_val1x2", "evaluation_valScala", "evaluation_valMetrica", "chance1x2_bookkeeping_status", "chance1x2_quote_diffInitialCurr2", "chance1x2_quote_diffInitialCurr1", "chance1x2_quote_current1"]
target_col = "win_1"

# Default LGBM params (puoi modificarli)
lgbm_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 12,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "objective": "binary",
    "metric": "precision",
}

# lgbm_params = {
#     "n_estimators": 500,
#     "learning_rate": 0.03,
#     "num_leaves": 16,
#     "max_depth": 4,
#     "min_data_in_leaf": 200,
#     "subsample": 0.7,
#     "subsample_freq": 1,
#     "colsample_bytree": 0.7,
#     "reg_lambda": 5.0,
#     "reg_alpha": 1.0,
#     "objective": "binary",
#     "metric": "auc",
#     "random_state": 42,
#     "n_jobs": -1,
# }
#
# lgbm_params = {
#     "n_estimators": 500,
#     "learning_rate": 0.05,
#     "num_leaves": 32,
#     "max_depth": 8,
#     "min_data_in_leaf": 200,
#     "subsample": 0.7,
#     "subsample_freq": 1,
#     "colsample_bytree": 0.7,
#     "reg_lambda": 5.0,
#     "reg_alpha": 1.0,
#     "objective": "binary",
#     "metric": "auc",
#     "random_state": 42,
#     "n_jobs": -1,
# }

# Load Data

In [3]:
set_mlflow_experiment(experiment_name=experiment_name)

query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Data Processing

## Train/Test/Val Split

In [127]:
# invece di lambda
def cast_float64(x):
    return x.astype("float64")

def bool_to_int_df(x):
    return bool_to_int(pd.DataFrame(x, columns=bool_cols))

def datetime_to_int64_df(x):
    return datetime_to_int64(pd.DataFrame(x, columns=datetime_cols))


In [128]:
unuseful_cols = ['team_league', 'team_home', 'team_away']

df = df_loaded.copy()
df = df.drop(unuseful_cols, axis=1)

# Basic sanity
df = df.dropna(how='all', axis=1)
# target must be 0/1
df[target_col] = df[target_col].astype(int)

num_cols, bool_cols, datetime_cols, cat_cols = infer_column_types(df, target_col)

train_df, val_df, test_df = time_based_split(df=df, time_col="time", train_frac=train_frac, val_frac=val_frac)

X_train = train_df[features_cols]
y_train = train_df[target_col].values

X_val = val_df[features_cols]
y_val = val_df[target_col].values

X_test = test_df[features_cols]
y_test = test_df[target_col].values

tot_cols = features_cols + [target_col]
num_cols = [x for x in num_cols if x in tot_cols]
bool_cols = [x for x in bool_cols if x in tot_cols]
datetime_cols = [x for x in datetime_cols if x in tot_cols]
cat_cols = [x for x in cat_cols if x in tot_cols]

# Preprocess
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", Pipeline(steps=[
#             ("astype", FunctionTransformer(lambda x: x.astype("float64"), validate=False)),
#             ]), num_cols),
#         ("bool", Pipeline(steps=[
#             ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=bool_cols), validate=False)),
#             ("cast", FunctionTransformer(bool_to_int, validate=False)),
#         ]), bool_cols),
#         ("dt", Pipeline(steps=[
#             ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=datetime_cols), validate=False)),
#             ("cast", FunctionTransformer(datetime_to_int64, validate=False)),
#         ]), datetime_cols),
#         # ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
#     ],
#     remainder="drop",
#     sparse_threshold=0.3,
#     verbose_feature_names_out=False,
# )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", FunctionTransformer(cast_float64, validate=False), num_cols),
        ("bool", FunctionTransformer(bool_to_int_df, validate=False), bool_cols),
        ("dt", FunctionTransformer(datetime_to_int64_df, validate=False), datetime_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=False,
)

model = LGBMClassifier(**lgbm_params)

# pipeline sklearn
pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", model),
])


In [129]:
with mlflow.start_run(run_name=run_name) as mlflow_run:
    # Log split info
    mlflow.log_params({
        "train_frac": train_frac,
        "val_frac": val_frac,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        **{f"lgbm__{k}": v for k, v in lgbm_params.items()},
    })

    # Fit with early stopping using validation
    # NB: early_stopping via fit params (LightGBM sklearn API)
    pipe.fit(
        X_train, y_train,
        clf__eval_set=[(preprocessor.fit_transform(X_val), y_val)],  # val transformed
        clf__eval_metric="auc",
        #clf__callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
        )

    # Predict proba
    p_train = pipe.predict_proba(X_train)[:, 1]
    p_val = pipe.predict_proba(X_val)[:, 1]
    p_test = pipe.predict_proba(X_test)[:, 1]

    pred_train = pipe.predict(X_train)
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)

    auc_train = roc_auc_score(y_train, p_train) if len(np.unique(y_train)) > 1 else np.nan
    auc_val = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan
    auc_test = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else np.nan

    precision_train = precision_score(y_train, pred_train)
    precision_val = precision_score(y_val, pred_val)
    precision_test = precision_score(y_test, pred_test)

    mlflow.log_metrics({
        "auc_train": float(auc_train) if np.isfinite(auc_train) else -1.0,
        "auc_val": float(auc_val) if np.isfinite(auc_val) else -1.0,
        "auc_test": float(auc_test) if np.isfinite(auc_test) else -1.0,
        "precision_train": float(precision_train) if np.isfinite(precision_train) else -1.0,
        "precision_val": float(precision_val) if np.isfinite(precision_val) else -1.0,
        "precision_test": float(precision_test) if np.isfinite(precision_test) else -1.0,
    })

    # Feature importance
    prep_fitted = pipe.named_steps["prep"]
    feature_names = get_feature_names_from_preprocessor(prep_fitted)
    

    booster = pipe.named_steps["clf"].booster_
    importances = booster.feature_importance(importance_type="gain")
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "importance_gain": importances
    }).sort_values("importance_gain", ascending=False)

    imp_csv = "feature_importance_gain.csv"
    mlflow.log_artifact(imp_csv)

    # Optional drop features under threshold & retrain
    if drop_importance_below > 0.0:
        keep_mask = imp_df["importance_gain"].values > drop_importance_below
        kept_features = imp_df.loc[keep_mask, "feature"].tolist()
        dropped = int((~keep_mask).sum())
        mlflow.log_params({
            "drop_importance_below": drop_importance_below,
            "dropped_features_count": dropped,
            "kept_features_count": len(kept_features),
        })

        # Per droppare in modo robusto con one-hot: selezioniamo colonne DOPO il preprocessor
        # Strategy: trasformiamo X_* e poi addestriamo un secondo LGBM su matrice ridotta.
        Xtr = prep_fitted.transform(X_train)
        Xva = prep_fitted.transform(X_val)
        Xte = prep_fitted.transform(X_test)

        keep_idx = np.where(keep_mask)[0]
        Xtr_k = Xtr[:, keep_idx]
        Xva_k = Xva[:, keep_idx]
        Xte_k = Xte[:, keep_idx]

        model2 = LGBMClassifier(**lgbm_params)
        model2.fit(
            Xtr_k, y_train,
            eval_set=[(Xva_k, y_val)],
            eval_metric="auc",
        )

        p_val2 = model2.predict_proba(Xva_k)[:, 1]
        p_test2 = model2.predict_proba(Xte_k)[:, 1]
        auc_val2 = roc_auc_score(y_val, p_val2) if len(np.unique(y_val)) > 1 else np.nan
        auc_test2 = roc_auc_score(y_test, p_test2) if len(np.unique(y_test)) > 1 else np.nan

        mlflow.log_metrics({
            "auc_val_dropped": float(auc_val2) if np.isfinite(auc_val2) else -1.0,
            "auc_test_dropped": float(auc_test2) if np.isfinite(auc_test2) else -1.0,
        })

        # Log modello ridotto come artifact “secondario”
        mlflow.lightgbm.log_model(model2, name="lgbm_model_retrained_after_drop")
        # Salviamo anche gli indici keep per riprodurre a runtime
        with open("artifacts/kept_feature_indices.json", "w") as f:
            json.dump(keep_idx.tolist(), f)
        mlflow.log_artifact("artifacts/kept_feature_indices.json")

    # Log modello pipeline (preprocess + lgbm)
    mlflow.sklearn.log_model(pipe, name="sklearn_pipeline_lgbm", input_example=X_train.dropna().iloc[:1])

    # ONNX export
    Xtr_trans = prep_fitted.transform(X_train)
    n_features_trans = Xtr_trans.shape[1]


    # Convert LightGBM booster to ONNX
    initial_types = [("input", FloatTensorType([None, n_features_trans]))]
    onnx_model = onnxmltools.convert_lightgbm(
        booster,
        initial_types=initial_types,
        target_opset=15,
    )

    with open(onnx_export_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    joblib.dump(preprocessor, preprocessor_export_path)
    print(f"Preprocessor salvato in {preprocessor_export_path}")

    mlflow.log_artifact(onnx_export_path)
    mlflow.log_artifact(preprocessor_export_path)

[LightGBM] [Info] Number of positive: 1817, number of negative: 2375
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000515 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1150
[LightGBM] [Info] Number of data points in the train set: 4192, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.433445 -> initscore=-0.267811
[LightGBM] [Info] Start training from score -0.267811
Preprocessor salvato in artifacts/preprocessor.joblib
🏃 View run test_1 at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/711814695936031/runs/9aff4f7df2124b0dada3ebf21a6ed501
🧪 View experiment at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/711814695936031


# Load ONNX Model

In [71]:
# 1. Carica il modello ONNX
sess = ort.InferenceSession(onnx_export_path)

# 2. Carica il preprocessor (su Raspberry Pi)
preprocessor = joblib.load(preprocessor_export_path)

def get_onnx_prediction(sess: InferenceSession, preprocessor: ColumnTransformer, input: pd.DataFrame, threshold: float=0.5):
    # Preprocess input
    x_trans = preprocessor.transform(input)  # trasformazione del preprocessor salvato
    x_trans = x_trans.astype(np.float32)        # ONNX richiede float32
    # ONNX Inference
    input_name = sess.get_inputs()[0].name
    outputs = sess.run(None, {input_name: x_trans})

    # Controlla quanti output ci sono e seleziona probabilità
    y_prob = np.array([x[1] for x in outputs[1]])  # se ONNX produce [label, probability]
    y_pred = np.array([int(x > threshold) for x in y_prob])

    return y_prob, y_pred

y_prob, y_pred = get_onnx_prediction(sess, preprocessor, X_test)

2026-02-08 17:24:12.730245 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {525} for output label


In [72]:
def check_onnx_predictions(y_prob: np.array, p_test:np.array):
    assert sum(np.round(y_prob,4) == np.round(p_test, 4)) == len(y_prob)

check_onnx_predictions(y_prob, p_test)

# Calculate Treshold and EV

In [73]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score
from sklearn.frozen import FrozenEstimator

odds_name = "chance1x2_quote_current1"

def get_threshold_and_test_ev(
    val_df, test_df,
    model, odds_name, target_col,
    P_MIN=0.75, N_MIN_PERC=0.1,
    EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.7, weight_nbets=0.3
):
    assert abs(weight_ev + weight_nbets - 1.0) < 1e-6

    X_val = val_df.drop(target_col, axis=1)
    y_val = val_df[target_col]
    X_test = test_df.drop(target_col, axis=1)
    y_test = test_df[target_col]


    # -----------------------------
    # 2. Calibration (fit on VAL)
    # -----------------------------
    calibrator = CalibratedClassifierCV(
        estimator=FrozenEstimator(model),
        method="isotonic",
        cv=5   # qui cv non serve per fare fit del model, ma per calibrare in CV (consigliato)
    )
    calibrator.fit(X_val, y_val)

    # 3. Predict on VAL
    p_val = calibrator.predict_proba(X_val)[:, 1]

    val_df["p_cal"] = p_val
    val_df["odds"] = val_df[odds_name]
    val_df["EV"] = val_df["p_cal"] * val_df["odds"] - 1

    # 4. Sweep thresholds (VAL)
    results = []
    THRESHOLDS = np.linspace(0.55, 0.90, 71)
    N_MIN = int(len(y_val) * N_MIN_PERC)

    for t in THRESHOLDS:
        sel = val_df[val_df["p_cal"] >= t]
        if len(sel) < N_MIN:
            continue

        precision = precision_score(
            sel[target_col].astype(int),
            np.ones(len(sel))
        )
        mean_ev = sel["EV"].mean()

        if precision < P_MIN or not (EV_MIN <= mean_ev <= EV_MAX):
            continue

        results.append({
            "threshold": t,
            "mean_EV_val": mean_ev,
            "precision_val": precision,
            "n_bets_val": len(sel),
            "n_bets_val_perc": int(len(sel) / len(y_val) * 100),
        })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        print("Nessuna soglia valida trovata")
        return None

    # 5. Score robusto (VAL)
    EV_CENTER = (EV_MIN + EV_MAX) / 2

    res_df["score"] = (
        weight_nbets * (res_df["n_bets_val"] / res_df["n_bets_val"].max()) +
        weight_ev * (
            1 - abs(res_df["mean_EV_val"] - EV_CENTER) / (EV_MAX - EV_MIN)
        )
    )

    optimal = res_df.sort_values("score", ascending=False).iloc[0]
    t_star = optimal["threshold"]

    # -----------------------------
    # 6. FINAL EVALUATION ON TEST
    # -----------------------------
    p_test = calibrator.predict_proba(X_test)[:, 1]

    test_df["p_cal"] = p_test
    test_df["odds"] = test_df[odds_name]
    test_df["EV"] = test_df["p_cal"] * test_df["odds"] - 1

    sel_test = test_df[test_df["p_cal"] >= t_star]

    test_metrics = {
        "threshold": t_star,
        "n_bets_test_perc": int(len(sel_test) / len(y_test) * 100),
        "mean_EV_test": sel_test["EV"].mean() if len(sel_test) > 0 else np.nan,
        "precision_test": precision_score(
            sel_test[target_col].astype(int),
            np.ones(len(sel_test))
        ) if len(sel_test) > 0 else np.nan
    }

    return {
        "val": optimal.to_dict(),
        "test": test_metrics
    }


optimal = get_threshold_and_test_ev(
    val_df, test_df, pipe, odds_name, target_col,
    P_MIN=0.75, EV_MIN=0.05, EV_MAX=0.5,
    weight_ev=0.5, weight_nbets=0.5, N_MIN_PERC=0.05
)
print(optimal)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
{'val': {'threshold': 0.7000000000000001, 'mean_EV_val': 0.09469230769230769, 'precision_val': 0.8461538461538461, 'n_bets_val': 26.

In [89]:
p_onnx = model.predict_proba(X_test)[:, 1]
quota = test_df[odds_name].to_numpy()

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200


In [103]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV

odds_val = val_df[odds_name].to_numpy()

# =========================
# 1. Expected Value
# =========================
def compute_ev(probs, y_true, odds, delta):
    """
    Bet se: p > 1 / odds + delta
    stake = 1
    """
    bet_mask = probs > (1.0 / odds + delta)

    if bet_mask.sum() == 0:
        return 0.0, 0.0

    wins = y_true[bet_mask] == 1
    losses = ~wins

    ev = (
        (odds[bet_mask][wins] - 1).sum()
        - losses.sum()
    )
    precision = wins.sum() / losses.sum()
    return ev, precision


# =========================
# 2. Calibrazione offline
# =========================
# lgbm_model: LGBMClassifier già fit su TRAIN

calibrator = CalibratedClassifierCV(
    model,
    method="sigmoid",
    cv="prefit"       # fondamentale
)

calibrator.fit(X_val, y_val)

# Probabilità calibrate (solo offline)
p_cal = calibrator.predict_proba(X_val)[:, 1]


# =========================
# 3. Ricerca di δ
# =========================
deltas = np.linspace(-0.05, 0.10, 300)
evs = []
precisions = []

for d in deltas:
    ev, precision = compute_ev(probs=p_cal, y_true=y_val, odds=odds_val, delta=d)
    evs.append(ev)
    precisions.append(precision)

df = pd.DataFrame({
    "delta": deltas,
    "ev": evs,
    "precision": precisions
})

# smoothing per stabilità
df["ev_smooth"] = df["ev"].rolling(15, center=True).mean()

best_delta = df.loc[df["ev_smooth"].idxmax(), "delta"]
precision = df.loc[df["ev_smooth"].idxmax(), "precision"]
best_ev = df["ev"].max()

print(f"Best δ = {best_delta:.4f} | EV = {best_ev:.2f} | Precision = {precision:.4f}")


# =========================
# 4. DEPLOY (Raspberry)
# =========================
# p_onnx = output predict_proba del modello ONNX
# quota = quota evento

def should_bet(p_onnx, quota, delta=best_delta):
    return p_onnx > (1.0 / quota + delta)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
Best δ = 0.0965 | EV = 0.27 | Precision = 0.3731


In [105]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score

odds_val = val_df[odds_name].to_numpy()

# =========================
# 1. Expected Value + Precision
# =========================
def compute_ev_and_precision(probs, y_true, odds, delta):
    """
    Bet se: p > 1 / odds + delta
    stake = 1
    Restituisce EV medio per bet e precision
    """
    bet_mask = probs > (1.0 / odds + delta)

    if bet_mask.sum() == 0:
        return 0.0, np.nan  # precision undefined

    wins = y_true[bet_mask] == 1
    losses = ~wins

    ev = (
        (odds[bet_mask][wins] - 1).sum()
        - losses.sum()
    ) / bet_mask.sum()  # media EV per bet

    precision = wins.sum() / bet_mask.sum()

    return ev, precision


# =========================
# 2. Calibrazione offline
# =========================
calibrator = CalibratedClassifierCV(
    model,
    method="sigmoid",
    cv="prefit"
)

calibrator.fit(X_val, y_val)

p_cal = calibrator.predict_proba(X_val)[:, 1]

# =========================
# 3. Ricerca di δ massimizzando precision
# =========================
deltas = np.linspace(-0.05, 0.10, 300)
results = []

for d in deltas:
    ev, precision = compute_ev_and_precision(
        probs=p_cal,
        y_true=y_val,
        odds=odds_val,
        delta=d
    )
    results.append({
        "delta": d,
        "ev": ev,
        "precision": precision,
        "n_bets": (p_cal > (1/odds_val + d)).sum()
    })

df = pd.DataFrame(results)

# smoothing opzionale
df["precision_smooth"] = df["precision"].rolling(15, center=True).mean()

# selezione della soglia che massimizza la precision
best_row = df.loc[df["precision_smooth"].idxmax()]
best_delta = best_row["delta"]
best_precision = best_row["precision"]
best_ev = best_row["ev"]

print(f"Best δ = {best_delta:.4f} | Precision = {best_precision:.2%} | EV medio = {best_ev:.2f}")

# =========================
# 4. DEPLOY (Raspberry)
# =========================
def should_bet(p_onnx, quota, delta=best_delta):
    return p_onnx > (1.0 / quota + delta)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
Best δ = -0.0425 | Precision = 32.87% | EV medio = -0.11


In [90]:
should_bet_col = np.array([should_bet(x, y) for x, y in zip(p_onnx, quota)])

In [91]:
should_bet_col.sum() / len(should_bet_col)

np.float64(0.35619047619047617)